# **Recommendation**

# 1. Build the User-Item Rating Matrix

In [2]:
import pandas as pd

master = pd.read_excel("../data/cleaned/master_dataset.xlsx")

# Create the user-item matrix: rows=UserId, columns=AttractionId, values=Rating
user_item_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating'
)

print("Matrix shape:", user_item_matrix.shape)
print(user_item_matrix.iloc[:5, :5])  # just peek at a small corner

Matrix shape: (33530, 30)
AttractionId  369  481   640  650  673
UserId                                
14            NaN  NaN  4.00  NaN  NaN
16            NaN  5.0  4.25  NaN  NaN
20            NaN  NaN   NaN  NaN  NaN
23            NaN  NaN   NaN  NaN  NaN
25            NaN  NaN   NaN  NaN  NaN


In [3]:
print("Unique attractions in transactions:", master['AttractionId'].nunique())
print("Unique users in transactions:", master['UserId'].nunique())

Unique attractions in transactions: 30
Unique users in transactions: 33530


# 2. (a)collaborativebased filtering-Compute similarity between users

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill NaN with 0 (0 = "no rating given", not "rated zero")
matrix_filled = user_item_matrix.fillna(0)

# Transpose so attractions become rows instead of users, then compute similarity
# This gives us a small 30x30 matrix instead of a massive 33530x33530 one
item_similarity = cosine_similarity(matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

print(item_similarity_df.shape)
print(item_similarity_df.iloc[:5, :5])

(30, 30)
AttractionId       369       481       640       650       673
AttractionId                                                  
369           1.000000  0.051794  0.071406  0.057954  0.067234
481           0.051794  1.000000  0.068480  0.043912  0.055698
640           0.071406  0.068480  1.000000  0.089273  0.089883
650           0.057954  0.043912  0.089273  1.000000  0.061950
673           0.067234  0.055698  0.089883  0.061950  1.000000


# 3. Generate recommendations for a sample user

In [5]:
def recommend_attractions(user_id, n_recommendations=5):
    # Get this user's ratings
    user_ratings = user_item_matrix.loc[user_id]
    
    # Which attractions has this user already rated?
    rated_attractions = user_ratings[user_ratings.notna()].index.tolist()
    
    # Score every attraction based on similarity to what the user already liked
    scores = pd.Series(0.0, index=user_item_matrix.columns)
    for attraction_id in rated_attractions:
        similarity_scores = item_similarity_df[attraction_id]
        scores += similarity_scores * user_ratings[attraction_id]
    
    # Remove attractions the user already visited (no point recommending those again)
    scores = scores.drop(rated_attractions)
    
    # Return the top N highest-scoring attractions
    return scores.sort_values(ascending=False).head(n_recommendations)

# Try it on a real user from our data
sample_user = user_item_matrix.index[0]
print(f"Recommendations for User {sample_user}:")
print(recommend_attractions(sample_user))

Recommendations for User 14:
AttractionId
749    1.717209
737    1.048467
824    0.952040
841    0.779154
673    0.755291
dtype: float64


# 4. show attraction names instead of just IDs

In [6]:
# Get unique attraction info from our master dataset (already has readable AttractionType)
attraction_info = master[['AttractionId', 'Attraction', 'AttractionType']].drop_duplicates()

def recommend_attractions_readable(user_id, n_recommendations=5):
    recommendations = recommend_attractions(user_id, n_recommendations)
    
    # Look up the attraction names for these IDs
    result = attraction_info[attraction_info['AttractionId'].isin(recommendations.index)].copy()
    result['Score'] = result['AttractionId'].map(recommendations)
    result = result.sort_values('Score', ascending=False)
    
    return result

# Test it on the same user, now with readable names
print(recommend_attractions_readable(14))

       AttractionId            Attraction   AttractionType     Score
36863           749  Tegenungan Waterfall       Waterfalls  1.717209
39053           737      Tanah Lot Temple  Religious Sites  1.048467
27689           824        Uluwatu Temple  Religious Sites  0.952040
13198           841         Waterbom Bali      Water Parks  0.779154
19627           673        Seminyak Beach          Beaches  0.755291


# 5. Split data for fair evaluation

In [17]:
from sklearn.model_selection import train_test_split

# Reload transaction data fresh (we need individual ratings, not the pivoted matrix)
transaction = pd.read_excel("../data/cleaned/Transaction_cleaned.xlsx")

# Split into train (80%) and test (20%) - same idea as Phases 5/6
train_data, test_data = train_test_split(transaction, test_size=0.2, random_state=42)

print("Train ratings:", len(train_data))
print("Test ratings:", len(test_data))

Train ratings: 42344
Test ratings: 10586


# 6. build the collaborative filtering matrix using ONLY training data

In [18]:
# Rebuild the user-item matrix using ONLY the training data
train_matrix = train_data.pivot_table(index='UserId', columns='AttractionId', values='Rating')
train_matrix_filled = train_matrix.fillna(0)

# Rebuild item similarity using ONLY training data
from sklearn.metrics.pairwise import cosine_similarity

train_item_similarity = cosine_similarity(train_matrix_filled.T)
train_item_similarity_df = pd.DataFrame(
    train_item_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

print("Train matrix shape:", train_matrix.shape)
print("Train item similarity shape:", train_item_similarity_df.shape)

Train matrix shape: (28700, 30)
Train item similarity shape: (30, 30)


# 7. Predict ratings for the test set

In [19]:
def predict_rating(user_id, attraction_id):
    # If user isn't in training data, or attraction isn't in training data, we can't predict
    if user_id not in train_matrix.index or attraction_id not in train_matrix.columns:
        return None
    
    user_ratings = train_matrix.loc[user_id]
    rated_attractions = user_ratings[user_ratings.notna()].index.tolist()
    
    if len(rated_attractions) == 0:
        return None
    
    # Weighted average: similarity-weighted rating from attractions this user already rated
    numerator = 0
    denominator = 0
    for rated_attraction in rated_attractions:
        similarity = train_item_similarity_df.loc[attraction_id, rated_attraction]
        rating = user_ratings[rated_attraction]
        numerator += similarity * rating
        denominator += abs(similarity)
    
    if denominator == 0:
        return None
    
    return numerator / denominator

# Quick test on one prediction
sample_user = test_data.iloc[0]['UserId']
sample_attraction = test_data.iloc[0]['AttractionId']
actual_rating = test_data.iloc[0]['Rating']

predicted = predict_rating(sample_user, sample_attraction)
print(f"User {sample_user}, Attraction {sample_attraction}")
print(f"Actual Rating: {actual_rating}")
print(f"Predicted Rating: {predicted}")

User 68981, Attraction 748
Actual Rating: 5
Predicted Rating: 3.0


# 8. Calculate RMSE across the full test set

In [20]:
import numpy as np

predictions = []
actuals = []

for idx, row in test_data.iterrows():
    pred = predict_rating(row['UserId'], row['AttractionId'])
    if pred is not None:  # skip cases where we couldn't make a prediction
        predictions.append(pred)
        actuals.append(row['Rating'])

print(f"Predictions made: {len(predictions)} out of {len(test_data)} test ratings")

# Calculate RMSE
rmse = np.sqrt(np.mean((np.array(predictions) - np.array(actuals)) ** 2))
mae = np.mean(np.abs(np.array(predictions) - np.array(actuals)))

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Predictions made: 5423 out of 10586 test ratings
RMSE: 1.0553
MAE: 0.6933


# 9. Save collaborative filtering artifacts

In [7]:
import joblib

joblib.dump(item_similarity_df, "../src/item_similarity.pkl")
joblib.dump(user_item_matrix, "../src/user_item_matrix.pkl")

print("Collaborative filtering artifacts saved!")

Collaborative filtering artifacts saved!


# 10. (b)Content-based filtering — using attraction features

In [8]:
item_full = pd.read_excel("../data/cleaned/Item_cleaned.xlsx")
attraction_type = pd.read_excel("../data/cleaned/Type_cleaned.xlsx")

# Merge to get readable attraction type for all 1,698 attractions
item_full = item_full.merge(attraction_type, on='AttractionTypeId', how='left')

print(item_full.shape)
print(item_full[['AttractionId', 'Attraction', 'AttractionType']].head())

(1698, 6)
   AttractionId                      Attraction           AttractionType
0           369               Kuta Beach - Bali                  Beaches
1           481                  Nusa Dua Beach                  Beaches
2           640  Sacred Monkey Forest Sanctuary  Nature & Wildlife Areas
3           650                     Sanur Beach                  Beaches
4           673                  Seminyak Beach                  Beaches


# 11. Build attraction feature profiles and compute similarity

In [9]:
# One-hot encode AttractionType - this becomes our "feature profile" for each attraction
attraction_features = pd.get_dummies(item_full[['AttractionId', 'AttractionType']], columns=['AttractionType'])

# Set AttractionId as index so it's easy to look up
attraction_features = attraction_features.set_index('AttractionId')

print(attraction_features.shape)
print(attraction_features.head())

(1698, 17)
              AttractionType_Ancient Ruins  AttractionType_Ballets  \
AttractionId                                                         
369                                  False                   False   
481                                  False                   False   
640                                  False                   False   
650                                  False                   False   
673                                  False                   False   

              AttractionType_Beaches  AttractionType_Caverns & Caves  \
AttractionId                                                           
369                             True                           False   
481                             True                           False   
640                            False                           False   
650                             True                           False   
673                             True                           Fal

# 12. Compute attraction similarity based on features

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity between all 1,698 attractions based on their type features
content_similarity = cosine_similarity(attraction_features)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=attraction_features.index,
    columns=attraction_features.index
)

print(content_similarity_df.shape)

(1698, 1698)


# 13. Build the content-based recommendation function

In [11]:
# Rebuild attraction_info using item_full (ALL 1698 attractions, not just the 30 with visit history)
attraction_info_full = item_full[['AttractionId', 'Attraction', 'AttractionType']].drop_duplicates()

def recommend_similar_attractions(attraction_id, n_recommendations=5):
    similar_scores = content_similarity_df[attraction_id]
    similar_scores = similar_scores.drop(attraction_id)
    top_similar = similar_scores.sort_values(ascending=False).head(n_recommendations)
    
    # Use attraction_info_full this time - works for ALL attractions, not just the 30
    result = attraction_info_full[attraction_info_full['AttractionId'].isin(top_similar.index)].copy()
    result = result.drop_duplicates(subset='AttractionId')
    result['SimilarityScore'] = result['AttractionId'].map(top_similar)
    result = result.sort_values('SimilarityScore', ascending=False)
    
    return result

# Test again
print(recommend_similar_attractions(640))

      AttractionId                               Attraction  \
19             975                             Sempu Island   
1666          2934                     Art Gallery - Monaco   
1683          2951           Farmers' Market - Isle of Mull   
1684          2952  Cultural Heritage Center - Isle of Mull   
1697          2965                 Botanical Garden - Sanaa   

               AttractionType  SimilarityScore  
19    Nature & Wildlife Areas              1.0  
1666          History Museums              0.0  
1683    Flea & Street Markets              0.0  
1684          History Museums              0.0  
1697           National Parks              0.0  


In [12]:
# How many attractions share the same type as attraction 640?
same_type = item_full[item_full['AttractionType'] == 'Nature & Wildlife Areas']
print(f"Total Nature & Wildlife Areas attractions: {len(same_type)}")
print(same_type[['AttractionId', 'Attraction']])

Total Nature & Wildlife Areas attractions: 2
    AttractionId                      Attraction
2            640  Sacred Monkey Forest Sanctuary
19           975                    Sempu Island


# 14. Add Country and Continent to attraction features

In [23]:
region = pd.read_excel("../data/cleaned/Region_cleaned.xlsx")

# Trace each attraction's location step by step, this time including Region
city = pd.read_excel("../data/cleaned/City_cleaned.xlsx")
country = pd.read_excel("../data/cleaned/Country_cleaned.xlsx")
continent = pd.read_excel("../data/cleaned/Continent_cleaned.xlsx")

item_location = item_full.merge(city, left_on='AttractionCityId', right_on='CityId', how='left')
item_location = item_location.merge(country, on='CountryId', how='left')
item_location = item_location.merge(region, on='RegionId', how='left')
item_location = item_location.merge(continent, on='ContinentId', how='left')

# Check the result
print(item_location[['AttractionId', 'Attraction', 'CityName', 'Country', 'Continent']].head())
print("\nMissing locations:", item_location['Country'].isnull().sum())

   AttractionId                      Attraction CityName   Country Continent
0           369               Kuta Beach - Bali   Douala  Cameroon    Africa
1           481                  Nusa Dua Beach   Douala  Cameroon    Africa
2           640  Sacred Monkey Forest Sanctuary   Douala  Cameroon    Africa
3           650                     Sanur Beach   Douala  Cameroon    Africa
4           673                  Seminyak Beach   Douala  Cameroon    Africa

Missing locations: 0


# 15. Evaluate Content-Based Filtering (precision-style)

In [21]:
# For users who rated MULTIPLE attractions of the SAME type, check consistency
master_check = master[['UserId', 'AttractionId', 'AttractionType', 'Rating']]

# Group by user + attraction type, only keep groups with 2+ ratings
grouped = master_check.groupby(['UserId', 'AttractionType'])['Rating'].agg(['count', 'std']).reset_index()
grouped = grouped[grouped['count'] >= 2]

print(f"Users with 2+ ratings in the same attraction type: {len(grouped)}")
print(f"Average rating standard deviation within same type: {grouped['std'].mean():.4f}")
print(f"(Compare to overall rating std: {master['Rating'].std():.4f})")

Users with 2+ ratings in the same attraction type: 6139
Average rating standard deviation within same type: 0.3932
(Compare to overall rating std: 0.9705)


# 16.  Document the type-based filtering

In [15]:
# Quick check on scope of the issue (for documentation purposes)
print("Attractions with AttractionCityId=1:", (item_full['AttractionCityId'] == 1).sum(), "out of", len(item_full))

Attractions with AttractionCityId=1: 14 out of 1698


# 17. Save content-based filtering artifacts

In [19]:
import joblib

joblib.dump(content_similarity_df, "../src/content_similarity.pkl")
joblib.dump(attraction_info_full, "../src/attraction_info_full.pkl")

print("Content-based filtering artifacts saved!")

Content-based filtering artifacts saved!
